# Results — Tables 4.6, 4.8, 4.10 and Figure 4.2
Attach the output of notebook 04 (all runs).

In [ ]:
# === STEP 0: SETUP — run this cell first, every time =========================
# Before running: in the right-hand panel
#   * "Session options" -> Accelerator = GPU T4 x2, Internet = On
#   * "Add Input" -> your code dataset (the uploaded bcd-project.zip)
#   * "Add Input" -> search "brain tumor mri dataset" by masoudnickparvar
#   * (notebooks 02-05) "Add Input" -> "Your Work" -> the previous notebook(s)
import os, sys, glob, shutil, subprocess

def find_dir(test, root='/kaggle/input', max_depth=6):
    # walk the attached inputs and return the first folder that passes test(folder)
    root_depth = root.rstrip('/').count('/')
    for d, subdirs, files in os.walk(root):
        if d.count('/') - root_depth > max_depth:
            subdirs[:] = []
            continue
        if test(d, subdirs, files):
            return d
    return None

# 1) the project code: the folder that contains bcd/__init__.py
code_dir = find_dir(lambda d, s, f: os.path.isfile(os.path.join(d, 'bcd', '__init__.py')))
if code_dir is None:
    raise SystemExit('Code not found. Click "Add Input" and attach the dataset you created from bcd-project.zip.')
shutil.copytree(code_dir, '/kaggle/working/bcd-project', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/bcd-project')
print('code found at:', code_dir)

# 2) install the extra libraries (takes ~1-2 minutes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<4.52', 'peft>=0.12',
                'bitsandbytes>=0.43', 'accelerate>=0.33', 'ultralytics', 'imagehash', 'h5py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])  # old torchao on Kaggle breaks peft

# 3) the brain MRI dataset: the folder that has Training/ and Testing/ inside
from bcd.config import load_config
cfg = load_config('/kaggle/working/bcd-project/configs/default.yaml')
mri = find_dir(lambda d, s, f: 'Training' in s and 'Testing' in s)
if mri is None:
    raise SystemExit('MRI dataset not found. Click "Add Input" and attach masoudnickparvar/brain-tumor-mri-dataset.')
cfg.paths.kaggle_root = mri
print('MRI dataset found at:', mri)

# 4) results of earlier notebooks (a folder named bcd that contains manifest.csv)
os.makedirs(cfg.paths.work_dir, exist_ok=True)
for prev in glob.glob('/kaggle/input/**/bcd/manifest.csv', recursive=True):
    src = os.path.dirname(prev)
    if not src.startswith('/kaggle/working'):
        shutil.copytree(src, cfg.paths.work_dir, dirs_exist_ok=True)
        print('restored earlier results from:', src)
print('GPUs available:', __import__('torch').cuda.device_count())
print('Setup complete. Work folder:', cfg.paths.work_dir)


In [ ]:
from bcd.report import tables, figure_rounds
W = cfg.paths.work_dir
t46, t48 = tables(f'{W}/runs', f'{W}/results')
t46

In [ ]:
t48

In [ ]:
figure_rounds(f'{W}/runs', f'{W}/results/figure_4_2.png')
from IPython.display import Image; Image(f'{W}/results/figure_4_2.png')

In [ ]:
# Table 4.10: latency and memory on this GPU
import time, torch, numpy as np, glob
from bcd.runtime import load_student, load_teacher, manifest, make_loader, part
from bcd.detector import TumourDetector
load = make_loader(cfg); paths = [p for p, _ in part(manifest(cfg), 'test')[:30]]
det = TumourDetector(glob.glob(f'{W}/yolo_runs/tumour/weights/best.pt')[0], device=0)
st = load_student(cfg, adapter=f'{W}/runs/P_seed0/adapter', device=0)
torch.cuda.reset_peak_memory_stats(0); t = []
for p in paths:
    s = time.time(); im = load(p); det(im); st.gradcam(im); t.append(time.time() - s)
print('regional pipeline s/case', np.mean(t), 'peak GB', torch.cuda.max_memory_allocated(0) / 1e9)
tc = load_teacher(cfg, adapter=f'{W}/adapters/teacher', device=1); torch.cuda.reset_peak_memory_stats(1); t = []
for p in paths[:10]:
    s = time.time(); tc.gradcam(load(p)); t.append(time.time() - s)
print('teacher s/case (excl. network)', np.mean(t), 'peak GB', torch.cuda.max_memory_allocated(1) / 1e9)